In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tarfile
import requests
import librosa as lb

**Preprocesamiento**

Primero se descarga la base de datos. El bloque para la descarga de la base de datos se asegura de no descargarla si esta ta existe, ya que al ser una base grande, tarda un tiempo considerable en descargarse. Luego se cargan y prueban 3 audios de las palabras "four", "go" y "house".

In [2]:
URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"

DATA_DIR = "data"
ARCHIVE_PATH = os.path.join(DATA_DIR, "speech_commands_v0.02.tar.gz")
EXTRACT_FLAG = os.path.join(DATA_DIR, ".extracted")  # archivo para saber si ya se extrajo

os.makedirs(DATA_DIR, exist_ok=True)

# Descargar si no existe
if not os.path.exists(ARCHIVE_PATH):
    r = requests.get(URL, stream=True)
    r.raise_for_status()
    with open(ARCHIVE_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# Extraer solo si no se ha extraído
if not os.path.exists(EXTRACT_FLAG):
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(path=DATA_DIR)
    # crear archivo flag
    open(EXTRACT_FLAG, "w").close()

In [3]:
four,sr_four = lb.load("data/four/0a2b400e_nohash_0.wav")
go,sr_go = lb.load("data/go/0a2b400e_nohash_0.wav")
house,sr_house = lb.load("data/house/0a2b400e_nohash_0.wav")

In [4]:
from IPython.display import Audio

In [5]:
Audio(four, rate = sr_four)

In [6]:
Audio(go, rate = sr_go)

In [7]:
Audio(house, rate = sr_house)

A continuación la función que se ocupa de calcular la stft de los fragmentos de audio, luego en base a las bandas de frecuengia genera 20 bandas de energía, con la siguiente formula:
$$E[i] = \sum_{k \in \mathcal{B}_{i}} |X[k]|^{2}$$
Luego esa energia se pasa a dB como:

$$E_{dB} = 10\log_{10}\left(\frac{E}{E_{máx}}\right)$$

In [8]:
def descomponer_en_energia(audio):
    stft = lb.stft(audio, n_fft=512, hop_length=256, win_length=512, window='hann')
    band_lenght = (stft.shape[0]//20)

    energy_bands = np.zeros([20, stft.shape[1]])

    stft_power = np.abs(stft)**2

    for i in range(20):
        for j in range(stft_power.shape[1]):
            if i == 19:
                energy_bands[i, j] = np.sum(stft_power[i*band_lenght:, j])
            else:
                energy_bands[i, j] = np.sum(stft_power[i*band_lenght:(i+1)*band_lenght, j])

    energy_max = energy_bands.max()
    return 10 * np.log10(np.maximum(energy_bands, 1e-12) / energy_max)

**Clasificación binaria**

Se implementará una regresón logistica binaria para reconocer los audios de "yes" y "no", para esto, primero se cargan los audios y luego se unifica su largo para que al descomponer en bandas de energía todos entreguen 50 ventanas de tiempo, el largo para que ocurra esto se obtiene de la siguiente manera:

$$L = (T - 1)\cdot \text{hop\_length}$$

Donde el hop_length es 256, obteniendo un largo de 12544

In [9]:

yes_dir = "data/yes/"

files_yes = [
    os.path.join(yes_dir, f)
    for f in os.listdir(yes_dir)
    if f.lower().endswith(".wav")
]

no_dir = "data/no/"

files_no = [
    os.path.join(no_dir, f)
    for f in os.listdir(no_dir)
    if f.lower().endswith(".wav")
]


In [10]:
def cargar_y_unificar(files):
    data_raw = np.zeros([len(files), 12544])
    for i in range(len(files)):
        y, sr = lb.load(files[i], sr=None)
        if len(y) < 12544:
            data_raw[i, :] = np.pad(y, (0, 12544 - len(y)))
        else:
            data_raw[i, :] = y[:12544]
    return data_raw

In [11]:
data_yes_raw = cargar_y_unificar(files_yes)
data_no_raw = cargar_y_unificar(files_no)

output_yes = np.ones(data_yes_raw.shape[0])
output_no = np.zeros(data_no_raw.shape[0])

print(data_yes_raw.shape)
print(data_no_raw.shape)

(4044, 12544)
(3941, 12544)


In [12]:
dataset_yes = np.zeros([data_yes_raw.shape[0], 20, 50])
dataset_no = np.zeros([data_no_raw.shape[0], 20, 50])

for i in range(data_yes_raw.shape[0]):
    dataset_yes[i, :, :] = descomponer_en_energia(data_yes_raw[i, :])

for i in range(data_no_raw.shape[0]):
    dataset_no[i, :, :] = descomponer_en_energia(data_no_raw[i, :])

print(dataset_yes.shape)
print(dataset_no.shape)

KeyboardInterrupt: 

Luego de obtener las bandas de energia de todas las muestras se estiran las matrices de la descomposicion en bandas de energia para que cada muestra quede como un unico vector de 1000 muestras.

In [ ]:
dataset_yes_plano = dataset_yes.reshape(dataset_yes.shape[0], -1)
dataset_no_plano = dataset_no.reshape(dataset_no.shape[0], -1)

print(dataset_yes_plano.shape)
print(dataset_no_plano.shape)

Por ultimo se mezcla cada set de datos por su cuenta, luego se dividen en los sets de entrenamiento y los de testeo, se concatenean los de "yes" y "no" y se vuelve a mezclar, para evitar cualquier tipo de sesgo de muestras en el entrenamiento/testeo

In [ ]:
rng = np.random.default_rng(42)

perm_yes = rng.permutation(dataset_yes_plano.shape[0])
perm_no = rng.permutation(dataset_no_plano.shape[0])

dataset_yes_mezclado = dataset_yes_plano[perm_yes]
output_yes_mezclado = output_yes[perm_yes]

dataset_no_mezclado = dataset_no_plano[perm_no]
output_no_mezclado = output_no[perm_no]

In [ ]:
split_yes = int(len(dataset_yes_mezclado) * 0.8)
split_no = int(len(dataset_no_mezclado) * 0.8)

train_yes = dataset_yes_mezclado[:split_yes]
test_yes = dataset_yes_mezclado[split_yes:]

train_out_yes = output_yes_mezclado[:split_yes]
test_out_yes = output_yes_mezclado[split_yes:]

train_no = dataset_no_mezclado[:split_no]
test_no = dataset_no_mezclado[split_no:]

train_out_no = output_no_mezclado[:split_no]
test_out_no = output_no_mezclado[split_no:]

train_set = np.concatenate((train_yes, train_no), axis=0)
train_output = np.concatenate((train_out_yes, train_out_no), axis=0)

test_set = np.concatenate((test_yes, test_no), axis=0)
test_output = np.concatenate((test_out_yes, test_out_no), axis=0)

In [ ]:
perm_train = rng.permutation(train_set.shape[0])
train_set = train_set[perm_train]
train_output = train_output[perm_train]

perm_test = rng.permutation(test_set.shape[0])
test_set = test_set[perm_test]
test_output = test_output[perm_test]

Para el entrenamiento se utilizará el gradiente de la media de la funcion log-loss, cuyo valor esperado es la cross entropy:

$$ J(w) = -\frac{1}{n} \sum_{i=1}^n \left[ y_i \log(h_i) + (1 - y_i)\log(1 - h_i) \right] $$

Donde $ h = \sigma(Xw), \quad \sigma(z) = \frac{1}{1 + e^{-z}} $. Se analizará un término de la sumatoria para simplicidad de cuentas, luego se vuelve a reemplazar en la sumatoria:

$$ \ell_i = -\left[ y_i \log(h_i) + (1 - y_i)\log(1 - h_i) \right] $$

Derivando respecto a w:

$$ \frac{\partial \ell_i}{\partial w}
= -\left[ \frac{y_i}{h_i}\frac{\partial h_i}{\partial w}
- \frac{1 - y_i}{1 - h_i}\frac{\partial h_i}{\partial w} \right] $$

$$ = -\left[ \left( \frac{y_i}{h_i} - \frac{1 - y_i}{1 - h_i} \right)
\frac{\partial h_i}{\partial w} \right] $$

Luego, calculando la derivada de $h_i$ y reemplazando:

$$ \frac{\partial h_i}{\partial w}
= h_i(1 - h_i)\, x_i $$

$$ \frac{\partial \ell_i}{\partial w}
= -\left[ \left( \frac{y_i}{h_i} - \frac{1 - y_i}{1 - h_i} \right)
h_i(1 - h_i)\, x_i \right] $$

Operando con el término entre corchetes se puede llegar a lo siguiente:

$$ \left( \frac{y_i}{h_i} - \frac{1 - y_i}{1 - h_i} \right)
h_i(1 - h_i)
= y_i(1 - h_i) - (1 - y_i)h_i $$

$$ = y_i - y_i h_i - h_i + y_i h_i $$

$$ = y_i - h_i $$

Por ultimo, se obtiene la derivada de un término de la sumatoria:

$$ \frac{\partial \ell_i}{\partial w}
= (h_i - y_i)x_i $$

Finalmente, reemplazando de nuevo en la sumatoria se obtiene el gradiente del riesgo empirico:

$$ \nabla_w J(w)
= \frac{1}{n} \sum_{i=1}^n (h_i - y_i)x_i $$

Esta ultima expresión se puede pasar a su forma matricial, obteniendo la formula que se utilizara finalmente en el codigo para el entrenamiento:

$$ = \frac{1}{n} X^T (h - y) $$

Cabe destacar que w tiene incluido el bias y x la columna de unos correspondiente para la suma del bias

In [ ]:
class BinaryLogisticRegression:
    # Opcional, para inicializar atributos o declarar hiperparámetros
    def __init__(self, iterations):
        self.params = None
        self.iterations = iterations
        self.learning_curve = np.zeros(self.iterations)
        self.mu = None
        self.sigma = None

    @staticmethod
    def _sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    @staticmethod
    def _add_bias(X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def _normalize(self, X):
        return (X - self.mu) / self.sigma

    # Etapa de entrenamiento
    def fit(self, X, y, rate):
        self.mu = np.mean(X, axis=0)
        self.sigma = np.std(X, axis=0)
        self.sigma[self.sigma == 0] = 1

        X = self._normalize(X)
        X = self._add_bias(X)

        N, d = X.shape
        self.params = np.ones(d)

        for i in range(self.iterations):
            y_soft = self._sigmoid(X @ self.params)
            gradient = (1/N) * X.T @ (y_soft - y)
            self.params = self.params - rate * gradient

            eps = 1e-15
            y_soft_clipped = np.clip(y_soft, eps, 1 - eps)

            loss = -np.mean(
                y * np.log(y_soft_clipped) +
                (1 - y) * np.log(1 - y_soft_clipped)
            )

            self.learning_curve[i] = loss

    # Etapa de testeo SOFT
    def predict_proba(self, X):
        X = self._normalize(X)
        X = self._add_bias(X)
        return self._sigmoid(X @ self.params)

    # Etapa de testeo HARD
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    # Cómputo del accuracy
    def accuracy(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

    # Cómputo de la entropía cruzada
    def cross_entropy(self, X, y):
        y_soft = self.predict_proba(X)
        eps = 1e-15
        y_soft = np.clip(y_soft, eps, 1 - eps)

        return -np.mean(
            y * np.log(y_soft) +
            (1 - y) * np.log(1 - y_soft)
        )

Cabe aclarar, que para que no se realizen cuentas que puedan arrojar nan por evaluar las funciones fuera de su doeminio, se clippea la entrada de la sigmoide y la prediccion soft que se hace en cada iteracion del entrenamiento

In [ ]:
binaryreg = BinaryLogisticRegression(iterations = 4000)

binaryreg.fit(train_set, train_output, rate = 0.1)

plt.plot(binaryreg.learning_curve)
plt.xlabel("Iteraciones")
plt.ylabel("Cross Entropy")
plt.title("Training Loss")
plt.grid()
plt.show()

Ajustando el learning rate y la cantidad de iteraciones de forma visual para que la cross entropy converja se opto por realizar 4000 iteraciones con un learning rate de 0.1 y se obtuvieron los siguientes valores de cross entropy y accuracy para el entrenamiento y el testeo

In [ ]:
print(f"Accuracy de entrenamiento:{binaryreg.accuracy(train_set, train_output)}")
print(f"Cross entropy de entrenamiento:{binaryreg.cross_entropy(train_set, train_output)}")

In [ ]:
print(f"Accuracy de testeo :{binaryreg.accuracy(test_set, test_output)}")
print(f"Cross entropy de testeo:{binaryreg.cross_entropy(test_set, test_output)}")

Los valores obtenidos se consideran buenos, y se mantienen similares tanto en entrenamiento como en testeo, la cross entropy se encuentra entre .1 y .2 y la accuracy es mayor a .9

Para visualizar el funcionamiento del algoritmo de otra forma, se arman las matrices de confusion, que muestran la proporcion de aciertos y fallas. Al ser una regresion binaria no es muy vistosa, pero en la regresion categorica de la proxima parte serán mas interesantes de analizar.

In [ ]:
y_pred = binaryreg.predict(train_set)

TP = np.sum((train_output == 1) & (y_pred == 1))
TN = np.sum((train_output == 0) & (y_pred == 0))
FP = np.sum((train_output == 0) & (y_pred == 1))
FN = np.sum((train_output == 1) & (y_pred == 0))

cm = np.array([[TN, FP],
               [FN, TP]])

# Normalizar por filas (proporciones)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

labels = ["No", "Yes"]

plt.imshow(cm_norm)

plt.xticks(range(2), labels)
plt.yticks(range(2), labels)

plt.title("Matriz de confusión")
plt.xlabel("Predicho")
plt.ylabel("Real")

for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")

plt.colorbar()
plt.show()

In [ ]:
y_pred = binaryreg.predict(test_set)

TP = np.sum((test_output == 1) & (y_pred == 1))
TN = np.sum((test_output == 0) & (y_pred == 0))
FP = np.sum((test_output == 0) & (y_pred == 1))
FN = np.sum((test_output == 1) & (y_pred == 0))

cm = np.array([[TN, FP],
               [FN, TP]])

# Normalizar por filas (proporciones)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

labels = ["No", "Yes"]

plt.imshow(cm_norm)

plt.xticks(range(2), labels)
plt.yticks(range(2), labels)

plt.title("Matriz de confusión")
plt.xlabel("Predicho")
plt.ylabel("Real")

for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")

plt.colorbar()
plt.show()

**Clasificación categórica**

Se reslizara una clasificacion categorica para los audios de "Up" "Down" "Right" y "Left", primero se preprocesan todos los datos de la misma forma que con la binaria, pero ahora con 4 sets:

In [ ]:

up_dir = "data/up/"

files_up = [
    os.path.join(up_dir, f)
    for f in os.listdir(up_dir)
    if f.lower().endswith(".wav")
]

down_dir = "data/down/"

files_down = [
    os.path.join(down_dir, f)
    for f in os.listdir(down_dir)
    if f.lower().endswith(".wav")
]

right_dir = "data/right/"

files_right = [
    os.path.join(right_dir, f)
    for f in os.listdir(right_dir)
    if f.lower().endswith(".wav")
]

left_dir = "data/left/"

files_left = [
    os.path.join(left_dir, f)
    for f in os.listdir(left_dir)
    if f.lower().endswith(".wav")
]

In [ ]:
data_up_raw = cargar_y_unificar(files_up)
data_down_raw = cargar_y_unificar(files_down)
data_right_raw = cargar_y_unificar(files_right)
data_left_raw = cargar_y_unificar(files_left)

output_up = 0*np.ones(data_up_raw.shape[0])
output_down = 1*np.ones(data_down_raw.shape[0])
output_right = 2*np.ones(data_right_raw.shape[0])
output_left= 3*np.ones(data_left_raw.shape[0])

print(data_up_raw.shape)
print(data_down_raw.shape)
print(data_right_raw.shape)
print(data_left_raw.shape)

In [ ]:
dataset_up = np.zeros([data_up_raw.shape[0], 20, 50])
dataset_down = np.zeros([data_down_raw.shape[0], 20, 50])
dataset_right = np.zeros([data_right_raw.shape[0], 20, 50])
dataset_left = np.zeros([data_left_raw.shape[0], 20, 50])

for i in range(data_up_raw.shape[0]):
    dataset_up[i, :, :] = descomponer_en_energia(data_up_raw[i, :])

for i in range(data_down_raw.shape[0]):
    dataset_down[i, :, :] = descomponer_en_energia(data_down_raw[i, :])

for i in range(data_right_raw.shape[0]):
    dataset_right[i, :, :] = descomponer_en_energia(data_right_raw[i, :])

for i in range(data_left_raw.shape[0]):
    dataset_left[i, :, :] = descomponer_en_energia(data_left_raw[i, :])

print(dataset_up.shape)
print(dataset_down.shape)
print(dataset_right.shape)
print(dataset_left.shape)

In [ ]:
dataset_up_plano = dataset_up.reshape(dataset_up.shape[0], -1)
dataset_down_plano = dataset_down.reshape(dataset_down.shape[0], -1)
dataset_right_plano = dataset_right.reshape(dataset_right.shape[0], -1)
dataset_left_plano = dataset_left.reshape(dataset_left.shape[0], -1)

print(dataset_up_plano.shape)
print(dataset_down_plano.shape)
print(dataset_right_plano.shape)
print(dataset_left_plano.shape)

In [ ]:
rng = np.random.default_rng(42)

perm_up = rng.permutation(dataset_up_plano.shape[0])
perm_down= rng.permutation(dataset_down_plano.shape[0])
perm_right = rng.permutation(dataset_right_plano.shape[0])
perm_left = rng.permutation(dataset_left_plano.shape[0])

dataset_up_mezclado = dataset_up_plano[perm_up]
output_up_mezclado = output_up[perm_up]

dataset_down_mezclado = dataset_down_plano[perm_down]
output_down_mezclado = output_down[perm_down]

dataset_right_mezclado = dataset_right_plano[perm_right]
output_right_mezclado = output_right[perm_right]

dataset_left_mezclado = dataset_left_plano[perm_left]
output_left_mezclado = output_left[perm_left]

In [ ]:
split_up = int(len(dataset_up_mezclado) * 0.8)
train_up = dataset_up_mezclado[:split_up]
test_up = dataset_up_mezclado[split_up:]
train_out_up = output_up_mezclado[:split_up]
test_out_up = output_up_mezclado[split_up:]


split_down = int(len(dataset_down_mezclado) * 0.8)
train_down = dataset_down_mezclado[:split_down]
test_down = dataset_down_mezclado[split_down:]
train_out_down = output_down_mezclado[:split_down]
test_out_down = output_down_mezclado[split_down:]

split_right = int(len(dataset_right_mezclado) * 0.8)
train_right = dataset_right_mezclado[:split_right]
test_right = dataset_right_mezclado[split_right:]
train_out_right = output_right_mezclado[:split_right]
test_out_right = output_right_mezclado[split_right:]

split_left = int(len(dataset_left_mezclado) * 0.8)
train_left = dataset_left_mezclado[:split_left]
test_left = dataset_left_mezclado[split_left:]
train_out_left = output_left_mezclado[:split_left]
test_out_left = output_left_mezclado[split_left:]

train_set_cat = np.concatenate((train_up, train_down, train_right, train_left), axis=0)
train_output_cat = np.concatenate((train_out_up, train_out_down, train_out_right, train_out_left), axis=0)

test_set_cat = np.concatenate((test_up, test_down, test_right, test_left), axis=0)
test_output_cat = np.concatenate((test_out_up, test_out_down, test_out_right, test_out_left), axis=0)

print(train_set_cat.shape)
print(train_output_cat.shape)
print(test_set_cat.shape)
print(test_output_cat.shape)

In [ ]:
perm_train = rng.permutation(train_set_cat.shape[0])
train_set_cat = train_set_cat[perm_train]
train_output_cat = train_output_cat[perm_train]

perm_test = rng.permutation(test_set_cat.shape[0])
test_set_cat = test_set_cat[perm_test]
test_output_cat = test_output_cat[perm_test]

Se utilizara un modelo softmax, basado en la operacion matematica de softmax:

$$softmax(k|a) = \frac{e^{a_k}}{\sum^K_{j = 1}e^{a_j}}$$

Nuevamente se utiliza como funcion costo a la log-loss, cuyo gradiente resulta:

$$ \nabla_W J(W) = \frac{1}{n} \sum_i x_i (\hat{y}_i - y_i)^T $$
$$ \nabla_W J(W) = \frac{1}{n} X^T (\hat{Y} - Y) $$

con $\hat{y}$ la softmax evaluada en $xW$

In [ ]:
class CategoricLogisticRegression:
    # Opcional, para inicializar atributos o declarar hiperparámetros
    def __init__(self, iterations):
        self.params = None
        self.iterations = iterations
        self.learning_curve = np.zeros(self.iterations)
        self.mu = None
        self.sigma = None
        self.K = None

    @staticmethod
    def _softmax(Z):
        Z = Z - np.max(Z, axis=1, keepdims=True)
        expZ = np.exp(Z)
        return expZ / np.sum(expZ, axis=1, keepdims=True)

    @staticmethod
    def _add_bias(X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def _normalize(self, X):
        return (X - self.mu) / self.sigma

    # Etapa de entrenamiento
    def fit(self, X, y, rate):
        y = y.astype(int)
        self.mu = np.mean(X, axis=0)
        self.sigma = np.std(X, axis=0)
        self.sigma[self.sigma == 0] = 1

        X = self._normalize(X)
        X = self._add_bias(X)

        N, d = X.shape
        self.K = np.max(y) + 1

        self.params = np.ones((d, self.K))

        Y = np.eye(self.K)[y]

        for i in range(self.iterations):
            Z = X @ self.params
            Y_hat = self._softmax(Z)

            gradient = (1/N) * X.T @ (Y_hat - Y)
            self.params = self.params - rate * gradient

            eps = 1e-15
            Y_hat_clipped = np.clip(Y_hat, eps, 1 - eps)

            loss = -np.mean(np.sum(Y * np.log(Y_hat_clipped), axis=1))
            self.learning_curve[i] = loss

    # Etapa de testeo SOFT
    def predict_proba(self, X):
        X = self._normalize(X)
        X = self._add_bias(X)
        return self._softmax(X @ self.params)

    # Etapa de testeo HARD
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    # Cómputo del accuracy
    def accuracy(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

    # Cómputo de la entropía cruzada
    def cross_entropy(self, X, y):
        y = y.astype(int)
        Y_hat = self.predict_proba(X)
        Y = np.eye(self.K)[y]

        eps = 1e-15
        Y_hat = np.clip(Y_hat, eps, 1 - eps)

        return -np.mean(np.sum(Y * np.log(Y_hat), axis=1))

**Aviso** La próxima celda en google colab tarda aprox 52 minutos, corriendolo localmente tarda menos de 30, de todas formas puede que la cantidad de iteraciones no sea necesaria. A partir de las 20mil iteraciones el decrecimiento de la cross entropy es de aproximadamente -0.01 cada 10mil iteraciones. Puede que para el alcance de este trabajo alcance con realizar menos iteraciones

In [ ]:
catreg = CategoricLogisticRegression(iterations = 30000)

print(train_set_cat.shape)
print(np.max(train_output_cat) + 1)

catreg.fit(train_set_cat, train_output_cat, rate = 0.025)

plt.plot(catreg.learning_curve)
plt.xlabel("Iteraciones")
plt.ylabel("Cross Entropy")
plt.title("Training Loss")
plt.grid()
plt.show()

Debajo se indican los parametros obtenidos luesgo del entrenamiento. En este caso las magnitudes no son tan buenas como con la clasificacion binaria, pero resulta esperable, ya que al tener mas clases para clasificar se tienen mas formas de fallar en la predicción.

In [ ]:
print(f"Accuracy de entrenamiento:{catreg.accuracy(train_set_cat, train_output_cat)}")
print(f"Cross entropy de entrenamiento:{catreg.cross_entropy(train_set_cat, train_output_cat)}")

In [ ]:
print(f"Accuracy de testeo:{catreg.accuracy(test_set_cat, test_output_cat)}")
print(f"Cross entropy de testeo:{catreg.cross_entropy(test_set_cat, test_output_cat)}")

Las matrices de confusion muestran el funcionamiento del algoritmo de forma grafica, se puede ver como la diagonal se encuantra resaltada, ya que en la diagonal se encuentran los exitos de la prediccion, y luego en las filas se pueden ver los casos en los que la prediccion no resultó acertada.

In [ ]:
y_pred = catreg.predict(train_set_cat)

train_output_cat = train_output_cat.astype(int)

K = 4
cm = np.zeros((K, K), dtype=int)

for i in range(len(train_output_cat)):
    cm[train_output_cat[i], y_pred[i]] += 1

# Normalizar por filas (proporciones)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

labels = ["Up", "Down", "Right", "Left"]

plt.imshow(cm_norm)

plt.xticks(range(K), labels)
plt.yticks(range(K), labels)

plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de confusión")

for i in range(K):
    for j in range(K):
        plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")

plt.colorbar()
plt.show()

# Métricas
acc = np.mean(y_pred == train_output_cat)

precision = []
sensibilidad = []

for k in range(K):
    TP = cm[k, k]
    FP = np.sum(cm[:, k]) - TP
    FN = np.sum(cm[k, :]) - TP

    p = TP / (TP + FP) if (TP + FP) > 0 else 0
    s = TP / (TP + FN) if (TP + FN) > 0 else 0

    precision.append(p)
    sensibilidad.append(s)

print(f"Accuracy (proporcion de aciertos): {acc:.4f}")
for i, label in enumerate(labels):
    print(f"{label} → Precisión: {precision[i]:.4f}, Sensibilidad: {sensibilidad[i]:.4f}")

In [ ]:
y_pred = catreg.predict(test_set_cat)

test_output_cat = test_output_cat.astype(int)

K = 4
cm = np.zeros((K, K), dtype=int)

for i in range(len(test_output_cat)):
    cm[test_output_cat[i], y_pred[i]] += 1

# Normalizar por filas (proporciones)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

labels = ["Up", "Down", "Right", "Left"]

plt.imshow(cm_norm)

plt.xticks(range(K), labels)
plt.yticks(range(K), labels)

plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de confusión")

for i in range(K):
    for j in range(K):
        plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")

plt.colorbar()
plt.show()

# Métricas
acc = np.mean(y_pred == test_output_cat)

precision = []
sensibilidad = []

for k in range(K):
    TP = cm[k, k]
    FP = np.sum(cm[:, k]) - TP
    FN = np.sum(cm[k, :]) - TP

    p = TP / (TP + FP) if (TP + FP) > 0 else 0
    s = TP / (TP + FN) if (TP + FN) > 0 else 0

    precision.append(p)
    sensibilidad.append(s)

print(f"Accuracy (proporcion de aciertos): {acc:.4f}")
for i, label in enumerate(labels):
    print(f"{label} → Precisión: {precision[i]:.4f}, Sensibilidad: {sensibilidad[i]:.4f}")